# Pushing world model

A small world model predicts, in DINOv2 feature space, where an object goes when a pusher moves. It is trained on
Push-T, then on 115 clips of my own hand pushing a box. My palm paths then drive a simulated Franka Panda.

**Runtime > Run all.** By default (the fast path) this notebook downloads the cached embeddings, trained predictors,
tracks and results from the Hugging Face dataset and rebuilds every figure and GIF. It needs no GPU and takes a few
minutes. Set `RETRAIN` below to retrain the predictors instead (needs a GPU: **Runtime > Change runtime type > T4 GPU**).

Push-T is downloaded from its original URL, never re-hosted.

In [ ]:
HF_DATASET = "well-iam/humanoid-intern-challenge"  # the Hugging Face dataset with my clips and the cached results
RETRAIN = False  # False: rebuild the figures from cached results (no GPU, a few minutes)
#                  "clips": embed my clips and retrain both predictors on them (T4: about 40 min)
#                  "all": also re-embed Push-T with DINOv2, retrain its two predictors and rerun the ranking test
#                         (T4: about 1.5 h). V-JEPA 2 and the extra seeds stay cached (commands in the README).

REPO_URL = "https://github.com/well-iam/humanoid-intern-challenge"
BRANCH = "main"

## Setup

Clones the repo and installs the pinned requirements on top of Colab's own PyTorch. Outside Colab it uses the
repo it runs in, and `PWM_DATA` (if set) as the data folder.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

assert RETRAIN in (False, "clips", "all"), RETRAIN
started = time.time()
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = Path("/content/humanoid-intern-challenge")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = Path(os.environ.get("PWM_DATA") or ("/content/pwm-data" if IN_COLAB else REPO / "data"))
GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
assert GPU or not RETRAIN, "RETRAIN needs a GPU: Runtime > Change runtime type > T4 GPU"

# The heavy work runs in the repo's stage scripts, in fresh processes: packages that pip just upgraded load
# their new versions without restarting the runtime.
env = {**os.environ, "PWM_DATA": str(DATA), "PYTHONPATH": str(REPO / "src")}
if IN_COLAB and not GPU:  # MuJoCo renders on the CPU instead (the Panda GIFs)
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y libosmesa6-dev > /dev/null", shell=True, check=True)
    env["MUJOCO_GL"] = "osmesa"


def run(script, *args, log=None):
    """Run one of the repo's stage scripts; print its output (and save it to `log`)."""
    print(f"$ python scripts/{script} {' '.join(args)}", flush=True)
    done = subprocess.run([sys.executable, str(REPO / "scripts" / script), *args], env=env, capture_output=True,
                          text=True)
    if log:
        Path(log).parent.mkdir(parents=True, exist_ok=True)
        Path(log).write_text(done.stdout)
    lines = done.stdout.strip().splitlines()
    print("\n".join(lines if len(lines) <= 12 else lines[:4] + ["..."] + lines[-7:]), flush=True)
    if done.returncode:
        raise RuntimeError(f"{script} failed:\n{done.stderr[-3000:]}")


commit = subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"], capture_output=True, text=True)
print(f"repo {REPO} at {commit.stdout.strip()}\ndata {DATA}\nGPU: {GPU}, Python {sys.version.split()[0]}")

## Download the dataset

The fast path takes everything the figures need (about 4.6 GB). It leaves out the full takes (720p videos, for
viewing), the V-JEPA 2 predictors and the predictors of the extra seeds. The V-JEPA 2 embeddings are not in
the dataset at all: V-JEPA 2 lost to DINOv2, and they are only needed to retrain it. With `RETRAIN`, the
embeddings and predictors that will be recomputed are not downloaded.

In [ ]:
import fnmatch

skip = ["clips/videos/*", "pusht/vjepa2/*.pt", "pusht/dinov2/seed-*/*.pt"]
if RETRAIN:
    skip += ["clips/dinov2/embeddings/*", "clips/dinov2/*.pt"]
if RETRAIN == "all":
    skip += ["pusht/dinov2/embeddings/*", "pusht/dinov2/*.pt"]

local = os.environ.get("PWM_HF_LOCAL")  # a local copy of the dataset, to test this notebook before uploading
if local:
    for path in sorted(Path(local).rglob("*")):
        name = path.relative_to(local).as_posix()
        if path.is_file() and not any(fnmatch.fnmatch(name, pattern) for pattern in skip):
            (DATA / name).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, DATA / name)
else:
    from huggingface_hub import snapshot_download

    snapshot_download(HF_DATASET, repo_type="dataset", local_dir=DATA, ignore_patterns=skip)
size = sum(p.stat().st_size for p in DATA.rglob("*") if p.is_file())
print(f"{DATA}: {size / 1e9:.1f} GB, {time.time() - started:.0f} s since the start")

## Push-T

Downloaded from its original URL (Chi et al., *Diffusion Policy*) and cut into the 150 episodes used (100 train,
50 held out).

In [ ]:
run("data.py", "--dataset", "pusht")

## Retrain (only with `RETRAIN`)

The fixed recipe of every run: DINOv2-small embeddings, a predictor of about 20M parameters, 2500 steps, learning
rate 1e-3, batch 16, fp16, seed 0. Each predictor trains next to an action-blind twin. The training logs feed
the training-curves figure.

In [ ]:
RECIPE = ["--precision", "fp16", "--batch-size", "16", "--steps", "2500", "--lr", "1e-3"]
LOGS = {"clips": ("clips/logs/train-predictor.log", "clips/logs/train-blind.log"),
        "pusht": ("pusht/logs/train-dinov2.log", "pusht/logs/train-dinov2-blind.log")}
if RETRAIN:
    for dataset in ["clips"] + (["pusht"] if RETRAIN == "all" else []):
        t = time.time()
        run("embed.py", "--dataset", dataset, "--encoder", "dinov2")
        run("train.py", "--dataset", dataset, "--encoder", "dinov2", *RECIPE, log=DATA / LOGS[dataset][0])
        run("train.py", "--dataset", dataset, "--encoder", "dinov2", *RECIPE, "--blind", log=DATA / LOGS[dataset][1])
        run("evaluate.py", "--dataset", dataset, "--encoder", "dinov2")
        if dataset == "clips":
            run("near_miss.py", "--dataset", dataset, "--encoder", "dinov2")
        else:
            run("compare.py", "--dataset", "pusht")
            try:  # the ranking test runs only if the validation gate passed
                run("rank.py", "--dataset", "pusht", "--encoder", "dinov2")
            except RuntimeError as error:
                print(error)
        print(f"{dataset}: retrained in {(time.time() - t) / 60:.0f} min")
else:
    print("RETRAIN is off: the cached predictors and results are used.")

## Rebuild every figure and GIF

From the saved results (error vs horizon, seeds, near-miss and ranking tests, probe error vs number of
training clips, training curves), then from the embeddings and predictors (retrievals: the nearest real frame
to each prediction), then in simulation (the Panda replays, on the CPU), and the GIF strip of all three. The tracking GIFs are shown as they come in the dataset: they need the full-size takes.

In [ ]:
figures_started = time.time()
run("redraw.py")
run("training_curves.py")
run("frame_strip.py", "--dataset", "pusht")
for dataset in ("pusht", "clips"):
    run("retrievals.py", "--dataset", dataset, "--encoder", "dinov2")
run("panda_replay.py", "--dataset", "pusht")
# Held-out clips: the median own-shift push, the median late-hand push, a close and a far near-miss (README)
run("panda_replay.py", "--dataset", "clips", "--episode", "VID_20261016_141304-11", "VID_20261016_163841-02",
    "VID_20261016_165020-05", "VID_20261016_141631-03")
run("gif_strip.py", "--episode", "VID_20261016_141304-11")
print(f"figures rebuilt in {time.time() - figures_started:.0f} s; {time.time() - started:.0f} s since the start")

## Results

My held-out clip, what the predictor expected one second earlier (shown as the nearest real frame from the
training clips), and the Panda replaying my palm path.

In [ ]:
from IPython.display import Image, Markdown, display


def show(*paths):
    for path in paths:
        display(Markdown(f"`{path}`"))
        display(Image(filename=str(DATA / path)))


show("clips/gif-strip/VID_20261016_141304-11.gif")

### Push-T

In [ ]:
show("pusht/frame-strip.png", "pusht/error-vs-horizon.png", "pusht/dinov2/error-vs-horizon-seeds.png",
     "pusht/dinov2/retrievals.png", "pusht/dinov2/ranking/ranking.png", "pusht/panda/pusht-002.gif")

### My clips

In [ ]:
show("clips/tracking/VID_20261016_141304.gif", "clips/dinov2/error-vs-horizon.png", "clips/dinov2/near-miss.png",
     "clips/dinov2/retrievals.png", "clips/dinov2/training-curves.png", "clips/dinov2/probe-scaling.png",
     "clips/panda/VID_20261016_141304-11.gif", "clips/panda/VID_20261016_163841-02.gif",
     "clips/panda/VID_20261016_165020-05.gif", "clips/panda/VID_20261016_141631-03.gif")
display(Markdown((DATA / "clips/dinov2/report.md").read_text(encoding="utf-8")))

### What was rebuilt

Every figure and GIF in the data folder, and whether this run made it.

In [ ]:
for path in sorted([*DATA.rglob("*.png"), *DATA.rglob("*.gif")]):
    made = "rebuilt" if path.stat().st_mtime >= started else "from the dataset"
    print(f"{made:17s} {path.relative_to(DATA).as_posix()}")
print(f"total {time.time() - started:.0f} s")